# Ebola2 validation

This notebook documents a replay of `ignore/Ebola2.zip`, a local
reference bundle containing input files, batch launch files, and
previous EbolaSim outputs for a single-administrative-unit North
Kivu run.

The zip is intentionally not committed. The committed evidence in
`docs/vignettes/ebola2/evidence` records how paramsets 1 and 2 were
rerun with this package, compares all 10,000 reference CSVs, and
stores a compact subset of reference/generated files for review.


In [ ]:
import csv
import importlib.util
import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("could not locate repository root")

ROOT = find_repo_root()
EVIDENCE = ROOT / "docs/vignettes/ebola2/evidence"
REPLAY_PATH = ROOT / "docs/vignettes/ebola2/replay_ebola2.py"
spec = importlib.util.spec_from_file_location("ebola2_replay_notebook", REPLAY_PATH)
replay = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = replay
spec.loader.exec_module(replay)

summary = json.loads((EVIDENCE / "comparison_summary.json").read_text(encoding="utf-8"))
{
    "overall_status": summary["overall_status"],
    "paramsets": summary["paramsets_requested"],
    "reference_files_checked": sum(
        item["comparison"]["files_expected"]
        for item in summary["paramsets"]
    ),
    "mismatches": sum(
        item["comparison"]["mismatch_count"]
        for item in summary["paramsets"]
    ),
}


When `ignore/Ebola2.zip` is present, the replay script extracts it
into an ignored work directory, reads the Windows batch files, and
rewrites only the output paths so reference outputs are never
overwritten.


In [ ]:
zip_path = ROOT / "ignore/Ebola2.zip"
if zip_path.is_file():
    extracted = replay.safe_extract_zip(
        zip_path,
        ROOT / "artifacts/ebola2-notebook/extracted",
    )
    bundle_root = replay.find_ebola_dir(extracted)
    print(f"Bundle root: {bundle_root}")
    batch_files = sorted(
        path.relative_to(bundle_root)
        for path in bundle_root.rglob("*.bat")
    )
    print("Batch files:")
    for path in batch_files:
        print(f"  {path}")
else:
    print(
        "ignore/Ebola2.zip is not present in this environment; "
        "using committed evidence."
    )


The generated command plans preserve `/P`, `/PP`, `/D`, `/S`, `/R`,
`/CLP1..13`, the four seeds, and `OMP_NUM_THREADS`. The only
intentional rewrite is `/O`, which points to an ignored generated
output directory.


In [ ]:
command_plans = []
for paramset in (1, 2):
    path = EVIDENCE / "commands" / f"paramset_{paramset}.json"
    command_plans.append(json.loads(path.read_text(encoding="utf-8")))

[
    {
        "paramset": plan["paramset"],
        "threads": plan["environment"]["OMP_NUM_THREADS"],
        "r_scale": next(arg for arg in plan["command"] if arg.startswith("/R:")),
        "first_clp": next(arg for arg in plan["command"] if arg.startswith("/CLP1:")),
        "last_clp": next(arg for arg in plan["command"] if arg.startswith("/CLP13:")),
        "seeds": plan["command"][-4:],
    }
    for plan in command_plans
]


The full comparison report checks every expected CSV. The subset
below recomputes direct byte/numeric comparisons for committed
reference/generated pairs: realisations 0 and 999, all five output
file types, for paramsets 1 and 2.


In [ ]:
subset_rows = []
for paramset in (1, 2):
    for realisation in replay.SELECTED_REALISATIONS:
        for suffix in replay.OUTPUT_SUFFIXES:
            filename = replay.paramset_output_name(paramset, realisation, suffix)
            reference = (
                EVIDENCE
                / "output_subset/reference"
                / f"paramset_{paramset}"
                / filename
            )
            generated = (
                EVIDENCE
                / "output_subset/generated"
                / f"paramset_{paramset}"
                / filename
            )
            comparison = replay.compare_output_file(
                reference,
                generated,
                tolerance=1e-9,
            )
            subset_rows.append(comparison.to_dict())

Counter(row["status"] for row in subset_rows)


In [ ]:
def read_series(path, x="t", y="incI"):
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
    return [float(row[x]) for row in rows], [float(row[y]) for row in rows]

fig, ax = plt.subplots(figsize=(8, 4))
for paramset in (1, 2):
    for kind, style in (("reference", "-"), ("generated", "--")):
        path = (
            EVIDENCE
            / f"output_subset/{kind}"
            / f"paramset_{paramset}"
            / f"paramset_{paramset}.0.csv"
        )
        t, inc = read_series(path, y="incI")
        ax.plot(t, inc, style, label=f"paramset {paramset} {kind}")

ax.set_xlabel("Day")
ax.set_ylabel("Incident infections")
ax.legend()
fig.tight_layout()
fig


To rerun the full validation locally, provide the zip and opt in to
the slow replay. This is deliberately not a default CI step.


In [ ]:
if os.environ.get("EBOLASIM_RUN_EBOLA2") == "1" and zip_path.is_file():
    exit_code = replay.main([
        "--zip", zip_path.as_posix(),
        "--workdir", (ROOT / "artifacts/ebola2-replay").as_posix(),
        "--evidence-dir", EVIDENCE.as_posix(),
        "--paramsets", "1", "2",
        "--threads", "4",
    ])
    print(f"Replay exit code: {exit_code}")
else:
    print(
        "Full replay skipped. Set EBOLASIM_RUN_EBOLA2=1 "
        "and provide ignore/Ebola2.zip."
    )
